# [검증] 진재언 위험지수 5변수 재현과 Cronbach's α — 2026-10-01

- 작성자: 이정연 (`ljy`)
- 목적: 충돌 #2(지수 산출 방법론) 검증 — 보고서 서술만으로 5변수를 재현해 Cronbach's α를 독립 계산
- 입력: `data/raw/sk_통신인구_202507~202512_통합.csv` (6개월, outer join 생성본)
- 전제: 집계구=BLOCK_CD 앞 13자리, 대상 시군구=강남구(11230)·춘천시(32010)
- **원 코드 미확인 — 각 변수 정의는 `src/features/risk_axes_v1.py` docstring의 가정을 따름**
- 다루지 않는 범위: 최종 위험지수 가중합, 지역 내부 백분위 정규화, 5분위 구간화

In [1]:
from pathlib import Path

import pandas as pd

from src.features.block_panel import aggregate_to_block
from src.features.risk_axes_v1 import (
    align_risk_direction,
    cronbach_alpha_standardized,
    reduce_to_block_level,
)

MONTHS = ["202507", "202508", "202509", "202510", "202511", "202512"]

repo_root = Path.cwd()
if not (repo_root / "data" / "raw").exists():
    repo_root = repo_root.parent
raw_dir = repo_root / "data" / "raw"

frames = [
    pd.read_csv(
        raw_dir / f"sk_통신인구_{month}_통합.csv",
        dtype={"STD_YM": "string", "BLOCK_CD": "string"},
    )
    for month in MONTHS
]
combined = pd.concat(frames, ignore_index=True)
print(f"격자×월 전체 행: {len(combined):,}")

panel = aggregate_to_block(combined)
print(f"집계구×월 패널 행: {len(panel):,}, 고유 집계구 수: {panel['집계구'].nunique():,}")

격자×월 전체 행: 721,318


집계구×월 패널 행: 9,834, 고유 집계구 수: 1,639


In [2]:
block_level = reduce_to_block_level(panel, months_order=MONTHS)
aligned = align_risk_direction(block_level)

print("집계구 단위 변수 요약:")
print(block_level.describe().round(3))

print("\n방향 정렬 후 상관행렬:")
print(aligned.corr().round(3))

alpha = cronbach_alpha_standardized(aligned)
print(f"\nCronbach's α (표준화, 상관행렬 기반): {alpha:.3f}")
print(f"항목 수: {aligned.shape[1]}, 유효 집계구 수: {aligned.dropna().shape[0]:,}")

집계구 단위 변수 요약:
            평탄도      야간비중       주말비      고령비중        추세
count  1639.000  1639.000  1639.000  1639.000  1639.000
mean      0.589     0.055     2.324     0.228    -0.011
std       0.053     0.013     4.031     0.054     0.033
min       0.427     0.020     0.074     0.108    -0.205
25%       0.553     0.046     0.761     0.190    -0.020
50%       0.582     0.054     1.167     0.229    -0.006
75%       0.624     0.063     2.197     0.258     0.005
max       0.901     0.114    68.116     0.575     0.211

방향 정렬 후 상관행렬:
         평탄도_위험  야간비중_위험  주말비_위험  고령비중_위험  추세_위험
평탄도_위험    1.000    0.838   0.286    0.205 -0.020
야간비중_위험   0.838    1.000   0.373    0.146 -0.051
주말비_위험    0.286    0.373   1.000    0.121  0.076
고령비중_위험   0.205    0.146   0.121    1.000  0.252
추세_위험    -0.020   -0.051   0.076    0.252  1.000

Cronbach's α (표준화, 상관행렬 기반): 0.589
항목 수: 5, 유효 집계구 수: 1,639


## 결론

- **Cronbach's α = 0.589** (0.7 미만 — 기준 미달)
- 집계구 단위(1,639개)는 재언 님 보고서의 "1,639개 × 6개월 = 9,834행"과 정확히 일치해, `집계구=BLOCK_CD 앞 13자리`라는 가정 자체는 신뢰할 만하다.
- 상관행렬을 보면 **평탄도_위험 ↔ 야간비중_위험이 0.838**로 사실상 같은 걸 재는 중복 변수다 — 재언 님 보고서 8장이 밝힌 "스피어만 상관 0.89로 이중가중" 문제와 방향·크기가 거의 같다. 보고서는 이 중복을 자동 감지해 가중치에서 하나를 뺐지만, 그 이후에도 **5개 전체의 α를 재검정한 기록은 없었다** — 지금 돌린 게 그 재검정이다.
- **추세_위험은 나머지와 거의 무관하다**(평탄도 -0.020, 야간비중 -0.051, 주말비 0.076) — 가중치가 가장 낮긴 하지만(0.118), 다른 4개와 같은 개념을 잰다고 보기 어렵다.
- **주말비_위험**은 평탄도·야간비중과 약한 양의 상관(0.29~0.37)은 있지만 고령비중·추세와는 거의 무관하다(0.12, 0.08) — 충돌 #3에서 의심했던 "방향이 맞는지 불확실하다"는 우려와 일관된 결과다.
- **결론**: 충돌 #2는 "역시 문제가 있었다" 쪽으로 기운다. 김태은 1차 설계(α −4.219)만큼 심각하진 않지만, 팀이 쓰는 0.7 기준에는 못 미친다. 다섯 변수를 그대로 가중합하기보다, (1) 평탄도·야간비중 중 하나만 남기고 (2) 추세를 제외하거나 재정의하는 쪽을 먼저 검토할 필요가 있다.
- **다음 확인 필요**: 이 계산 자체가 원 코드 미확인 상태의 재구성이므로, 재언 님 실제 코드가 공유되면 (a) 집계구 정의, (b) 주말비 방향, (c) 평탄도/야간비중/추세의 정확한 공식을 대조해 이 α 값을 재검증해야 한다.
